# Practical Statistics for Data Scientists (Python)
# Chapter 4 — Regression and Prediction

> **Author:** Satrio Arjuna Putra  
> **Reference:** *Practical Statistics for Data Scientists* — Bruce, Bruce & Gedeck (O'Reilly)

---

## 📋 Chapter Summary

Chapter 4 covers **regression** — modelling the relationship between a response variable and one or more predictors.

| Topic | Key Concepts |
|-------|-------------|
| **Simple Linear Regression** | One predictor; OLS; regression equation |
| **Multiple Linear Regression** | Multiple predictors; interpretation; multicollinearity |
| **Model Assessment** | R², RMSE, residual analysis |
| **Model Selection** | Forward/backward stepwise; AIC |
| **Weighted Regression** | Handle heteroscedasticity |
| **Factor Variables** | Encoding categorical predictors |
| **Interaction Terms** | When effect of X₁ depends on X₂ |
| **Polynomial Regression** | Non-linear relationships |
| **Partial Residual Plots** | Visualise individual predictor relationships |
| **Correlated Predictors** | Multicollinearity effects |

---


## 🔧 Setup & Imports

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import statsmodels.api as sm
import statsmodels.formula.api as smf

DATA_BASE = "https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/master/data/"
HOUSE_CSV = DATA_BASE + "house_sales.csv"

np.random.seed(42)
print("Libraries loaded!")


In [ ]:
# Load house sales data
house = pd.read_csv(HOUSE_CSV, sep='\t')
print("Shape:", house.shape)
print("\nColumns:", list(house.columns))
print("\nSample data:")
house.head()


---

## 1. Simple Linear Regression

### 📚 Theory

**Simple Linear Regression** models the relationship between one predictor X and one response Y:

$$Y = \beta_0 + \beta_1 X + \epsilon$$

- **β₀ (intercept):** Value of Y when X = 0
- **β₁ (slope):** Change in Y for a 1-unit increase in X
- **ε (error):** Random noise

**Ordinary Least Squares (OLS)** minimises the sum of squared residuals:
$$\min_{\beta_0, \beta_1} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

> **Key Insight:** OLS gives the best linear unbiased estimator (BLUE) under standard assumptions (Gauss-Markov theorem).


In [ ]:
# Simple regression: SqFtTotLiving → AdjSalePrice
X = house[['SqFtTotLiving']]
y = house['AdjSalePrice']

model = LinearRegression()
model.fit(X, y)

print(f"Intercept (β₀): {model.intercept_:,.2f}")
print(f"Slope (β₁):     {model.coef_[0]:,.2f}")
print(f"R²:             {model.score(X, y):.4f}")
print()
print("Interpretation:")
print(f"  Each additional sq ft adds ~${model.coef_[0]:,.0f} to house price")


In [ ]:
# Visualise simple regression
plt.figure(figsize=(8, 5))
plt.scatter(house['SqFtTotLiving'], house['AdjSalePrice'],
            alpha=0.3, s=10, label='Data')
x_range = np.linspace(house['SqFtTotLiving'].min(), house['SqFtTotLiving'].max(), 100)
y_pred  = model.intercept_ + model.coef_[0] * x_range
plt.plot(x_range, y_pred, 'r-', linewidth=2, label='Regression Line')
plt.xlabel('Living Area (sq ft)')
plt.ylabel('Adjusted Sale Price ($)')
plt.title('Simple Linear Regression: Living Area vs Sale Price')
plt.legend()
plt.tight_layout()
plt.show()


---

## 2. Multiple Linear Regression

### 📚 Theory

**Multiple Linear Regression** extends to multiple predictors:

$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \ldots + \beta_p X_p + \epsilon$$

**Key considerations:**
- **Interpretation:** β_j = change in Y per 1-unit increase in X_j, *holding other variables constant*
- **Multicollinearity:** Highly correlated predictors make coefficients unstable (check VIF)
- **Feature scaling:** Not required for OLS, but helps with interpretation

> **Key Insight:** In multiple regression, each coefficient represents the *partial* effect — the effect of one variable after controlling for all others.


In [ ]:
# Multiple regression with statsmodels (for full summary)
features = ['SqFtTotLiving', 'SqFtLot', 'Bathrooms', 'Bedrooms', 'BldgGrade']
X = house[features]
y = house['AdjSalePrice']

# Add intercept
X_sm = sm.add_constant(X)
model_sm = sm.OLS(y, X_sm).fit()
print(model_sm.summary())


In [ ]:
# Sklearn multiple regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

model_ml = LinearRegression()
model_ml.fit(X, y)

coef_df = pd.DataFrame({'Feature': features, 'Coefficient': model_ml.coef_})
print("Coefficients:")
print(coef_df.to_string(index=False))
print(f"\nR²: {model_ml.score(X, y):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y, model_ml.predict(X))):,.0f}")


---

## 3. Model Assessment

### 📚 Theory

**Key metrics for regression:**

| Metric | Formula | Notes |
|--------|---------|-------|
| **R²** | $1 - SS_{res}/SS_{tot}$ | Proportion of variance explained (0–1) |
| **Adjusted R²** | Penalises for number of predictors | Better for model comparison |
| **RMSE** | $\sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$ | In same units as Y |
| **MAE** | $\frac{1}{n}\sum|y_i - \hat{y}_i|$ | Robust to outliers |

**Residual Analysis:** Residuals = $y - \hat{y}$ should be:
- Normally distributed
- Zero-centred
- No pattern against fitted values (homoscedastic)

> **Key Insight:** R² alone is insufficient. Always examine residual plots to validate model assumptions.


In [ ]:
# Residual analysis
y_pred = model_ml.predict(X)
residuals = y - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Residuals vs Fitted
axes[0].scatter(y_pred, residuals, alpha=0.3, s=5)
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Fitted Values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

# Histogram of Residuals
axes[1].hist(residuals, bins=50)
axes[1].set_xlabel('Residuals')
axes[1].set_title('Distribution of Residuals')

plt.tight_layout()
plt.show()

print(f"R²:   {r2_score(y, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y, y_pred)):,.0f}")


---

## 4. Factor Variables (Categorical Predictors)

### 📚 Theory

Categorical variables must be **encoded** before use in regression:

**One-hot encoding (dummy variables):**  
For a variable with k categories, create k-1 binary indicators (drop one to avoid multicollinearity — the "dummy variable trap").

**Interpretation:** Each dummy coefficient represents the difference in Y relative to the **reference category**.

> **Key Insight:** Always drop one category (or use `drop_first=True`). The choice of reference category affects coefficient interpretation but not model fit.


In [ ]:
# Factor variable: PropertyType as dummy variables
house_encoded = pd.get_dummies(house, columns=['PropertyType'], drop_first=True)
factor_cols = ['SqFtTotLiving', 'SqFtLot', 'Bathrooms', 'Bedrooms', 'BldgGrade'] +               [c for c in house_encoded.columns if c.startswith('PropertyType_')]

X_factor = house_encoded[factor_cols]
y = house['AdjSalePrice']

model_factor = LinearRegression()
model_factor.fit(X_factor, y)

coef_df = pd.DataFrame({'Feature': factor_cols, 'Coefficient': model_factor.coef_})
print(coef_df.to_string(index=False))
print(f"\nR²: {model_factor.score(X_factor, y):.4f}")


---

## 5. Interpreting Regression Results

### 📚 Theory

Beyond coefficients, regression output includes:

- **Standard Error:** Uncertainty in the coefficient estimate
- **t-statistic:** coefficient / SE (is the coefficient distinguishable from 0?)
- **p-value:** Evidence against H₀: β = 0
- **Confidence Interval:** Plausible range for the true coefficient

**Multicollinearity Check:**
- **Variance Inflation Factor (VIF):** VIF > 5–10 indicates problematic collinearity
- High VIF → coefficients become unstable, standard errors inflate

> **Key Insight:** High p-value doesn't mean a variable is unimportant — it could be a multicollinearity issue. Always check VIF alongside p-values.


In [ ]:
# Detailed regression summary with statsmodels
X_sm = sm.add_constant(house_encoded[factor_cols])
model_detailed = sm.OLS(y, X_sm).fit()

# Show coefficient table
coef_summary = model_detailed.summary2().tables[1]
print(coef_summary.round(4))


In [ ]:
# VIF (Variance Inflation Factor) for multicollinearity
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data['Feature'] = factor_cols
vif_data['VIF'] = [variance_inflation_factor(X_factor.values, i)
                   for i in range(X_factor.shape[1])]
print("VIF (< 5 = acceptable):")
print(vif_data.sort_values('VIF', ascending=False).to_string(index=False))


---

## 6. Polynomial Regression and Interaction Terms

### 📚 Theory

**Polynomial Regression** captures non-linear relationships:
$$Y = \beta_0 + \beta_1 X + \beta_2 X^2 + \ldots + \epsilon$$

**Interaction Terms** model when the effect of X₁ on Y depends on X₂:
$$Y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \beta_3 X_1 X_2 + \epsilon$$

> **Key Insight:** Polynomial regression is still *linear* in the coefficients — just uses transformed features. Overfitting is a risk with high-degree polynomials.


In [ ]:
# Polynomial regression: SqFtTotLiving with degree 2
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(house[['SqFtTotLiving']])

model_poly = LinearRegression()
model_poly.fit(X_poly, y)

# Plot
x_range = np.linspace(house['SqFtTotLiving'].min(), house['SqFtTotLiving'].max(), 200).reshape(-1, 1)
x_poly_range = poly.transform(x_range)
y_poly_pred = model_poly.predict(x_poly_range)

plt.figure(figsize=(8, 5))
plt.scatter(house['SqFtTotLiving'], house['AdjSalePrice'], alpha=0.2, s=5, label='Data')
plt.plot(x_range, y_poly_pred, 'r-', linewidth=2, label='Polynomial (degree=2)')
plt.xlabel('Living Area (sq ft)')
plt.ylabel('Adjusted Sale Price ($)')
plt.title('Polynomial Regression: Living Area vs Sale Price')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Polynomial R²: {model_poly.score(X_poly, y):.4f}")


---

## 7. Model Selection with AIC/BIC

### 📚 Theory

**Information criteria** balance model fit against complexity:

$$AIC = 2k - 2\ln(\hat{L})$$
$$BIC = k\ln(n) - 2\ln(\hat{L})$$

Where k = number of parameters, n = sample size, L = likelihood.

**Lower AIC/BIC = better model.**

**Stepwise selection:**
- **Forward:** Start with no variables; add one at a time (best AIC)
- **Backward:** Start with all variables; remove one at a time

> **Key Insight:** AIC/BIC prevent overfitting by penalising model complexity. BIC penalises more heavily for additional parameters.


In [ ]:
# Compare models with AIC using statsmodels
features_sets = {
    'Model 1 (1 var)': ['SqFtTotLiving'],
    'Model 2 (3 vars)': ['SqFtTotLiving', 'Bathrooms', 'Bedrooms'],
    'Model 3 (5 vars)': ['SqFtTotLiving', 'SqFtLot', 'Bathrooms', 'Bedrooms', 'BldgGrade'],
}

results = []
for name, feats in features_sets.items():
    X_sm = sm.add_constant(house[feats])
    m = sm.OLS(y, X_sm).fit()
    results.append({'Model': name, 'R²': m.rsquared, 'Adj R²': m.rsquared_adj,
                    'AIC': m.aic, 'BIC': m.bic, 'N params': len(feats) + 1})

print(pd.DataFrame(results).to_string(index=False))


---

## ✅ Chapter 4 Key Takeaways

1. **Simple regression:** β₁ = change in Y per 1-unit increase in X (caution: not causal without randomisation).
2. **Multiple regression:** Each β = partial effect, controlling for all other variables.
3. **R² and RMSE:** Always report both and inspect residual plots.
4. **Categorical variables:** Use one-hot encoding; choose a meaningful reference category.
5. **Multicollinearity:** Check VIF; high VIF makes coefficients unstable.
6. **Polynomial/Interaction terms:** Capture non-linearity and interaction effects.
7. **Model selection:** Use AIC/BIC to balance fit and complexity.

---
*Reference: Chapter 4, Practical Statistics for Data Scientists (O'Reilly)*
